<a href="https://colab.research.google.com/github/SofiiaBobr/goit_machine_learning/blob/main/hw5_olist.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Домашнє завдання №5 — CTE, Recursive CTE, Materialized Views
**Дисципліна:** Relational Databases for DS · **Датасет:** Olist Brazilian E-Commerce · **Автор:** Софія Бобрівець

Цей notebook містить повний відтворюваний pipeline із typed-схемою, CTE-chain, рекурсивним графом, materialized view, concurrent refresh, EXPLAIN (ANALYZE, BUFFERS) і автоматично сформованим Reflection. Усі SQL-запити виконуються на PostgreSQL, а не в SQLite. Використовується **повний Olist**, не stub.

**Запуск:** Google Colab → Runtime → Restart session and run all. Інтернет потрібний для встановлення пакетів і першого завантаження KaggleHub. Якщо KaggleHub недоступний, покладіть CSV до `data/olist/`. Результати EXPLAIN і Reflection будуть згенеровані лише після реального виконання.

## Завдання 1. Setup PostgreSQL, raw → typed, smoke tests

In [1]:
%pip install -q pgserver psycopg2-binary sqlalchemy pandas kagglehub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 40.0 MB/s eta 0:00:00


In [2]:
import os, pathlib, time, re
import pandas as pd
import pgserver
from sqlalchemy import create_engine, text
from IPython.display import display, Markdown

pg = pgserver.get_server('/tmp/hw5_pg', cleanup_mode='stop')
engine = create_engine(pg.get_uri(), future=True)
with engine.connect() as conn:
    print('PostgreSQL:', conn.execute(text('SELECT version()')).scalar_one())

def query(sql, params=None):
    with engine.connect() as conn:
        return pd.read_sql_query(text(sql), conn, params=params)

def execute(sql):
    with engine.begin() as conn:
        return conn.execute(text(sql)).rowcount

def show(sql, n=10):
    df=query(sql)
    display(df.head(n))
    print(f'Показано {min(len(df),n)} із {len(df)} рядків')
    return df

PostgreSQL: PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


In [3]:
import kagglehub
local_dir = pathlib.Path('data/olist')
required = {
    'olist_customers_raw':'olist_customers_dataset.csv',
    'olist_orders_raw':'olist_orders_dataset.csv',
    'olist_order_items_raw':'olist_order_items_dataset.csv',
    'olist_products_raw':'olist_products_dataset.csv',
    'olist_sellers_raw':'olist_sellers_dataset.csv',
}
if all((local_dir / f).is_file() for f in required.values()):
    dataset_dir = local_dir
    print('Джерело: локальна папка', dataset_dir)
else:
    dataset_dir = pathlib.Path(kagglehub.dataset_download('olistbr/brazilian-ecommerce'))
    print('Джерело: KaggleHub', dataset_dir)
for table, filename in required.items():
    path = dataset_dir / filename
    if not path.is_file():
        matches = list(dataset_dir.rglob(filename))
        if not matches: raise FileNotFoundError(f'Не знайдено {filename} у {dataset_dir}')
        path = matches[0]
    df = pd.read_csv(path, low_memory=False)
    df.to_sql(table, engine, if_exists='replace', index=False, chunksize=5000)
    print(f'{table}: {len(df):,} rows, {len(df.columns)} columns')

Джерело: KaggleHub /kaggle/input/brazilian-ecommerce
olist_customers_raw: 99,441 rows, 5 columns
olist_orders_raw: 99,441 rows, 8 columns
olist_order_items_raw: 112,650 rows, 7 columns
olist_products_raw: 32,951 rows, 9 columns
olist_sellers_raw: 3,095 rows, 4 columns


In [4]:
SCHEMA_SQL = "DROP MATERIALIZED VIEW IF EXISTS mv_olist_customer_features CASCADE;\nDROP TABLE IF EXISTS hw5_referrals CASCADE;\nDROP TABLE IF EXISTS olist_order_items CASCADE;\nDROP TABLE IF EXISTS olist_orders CASCADE;\nDROP TABLE IF EXISTS olist_customers CASCADE;\nDROP TABLE IF EXISTS olist_products CASCADE;\nDROP TABLE IF EXISTS olist_sellers CASCADE;\nCREATE TABLE olist_customers (\n customer_id TEXT PRIMARY KEY, customer_unique_id TEXT NOT NULL,\n customer_zip_code_prefix INTEGER, customer_city TEXT, customer_state CHAR(2) NOT NULL\n);\nCREATE INDEX idx_hw5_customer_unique ON olist_customers(customer_unique_id);\nCREATE TABLE olist_orders (\n order_id TEXT PRIMARY KEY, customer_id TEXT NOT NULL REFERENCES olist_customers(customer_id),\n order_status TEXT NOT NULL, order_purchase_timestamp TIMESTAMP NOT NULL,\n order_approved_at TIMESTAMP, order_delivered_carrier_date TIMESTAMP,\n order_delivered_customer_date TIMESTAMP, order_estimated_delivery_date TIMESTAMP,\n CHECK(order_status IN ('created','approved','invoiced','processing','shipped','delivered','unavailable','canceled'))\n);\nCREATE INDEX idx_hw5_orders_customer_ts ON olist_orders(customer_id,order_purchase_timestamp);\nCREATE TABLE olist_products (\n product_id TEXT PRIMARY KEY, product_category_name TEXT,\n product_name_length INTEGER CHECK(product_name_length IS NULL OR product_name_length>=0),\n product_description_length INTEGER CHECK(product_description_length IS NULL OR product_description_length>=0),\n product_photos_qty INTEGER CHECK(product_photos_qty IS NULL OR product_photos_qty>=0),\n product_weight_g INTEGER CHECK(product_weight_g IS NULL OR product_weight_g>=0),\n product_length_cm INTEGER CHECK(product_length_cm IS NULL OR product_length_cm>=0),\n product_height_cm INTEGER CHECK(product_height_cm IS NULL OR product_height_cm>=0),\n product_width_cm INTEGER CHECK(product_width_cm IS NULL OR product_width_cm>=0)\n);\nCREATE TABLE olist_sellers (\n seller_id TEXT PRIMARY KEY, seller_zip_code_prefix INTEGER, seller_city TEXT, seller_state CHAR(2) NOT NULL\n);\nCREATE TABLE olist_order_items (\n order_id TEXT NOT NULL REFERENCES olist_orders(order_id),\n order_item_id INTEGER NOT NULL, product_id TEXT REFERENCES olist_products(product_id),\n seller_id TEXT REFERENCES olist_sellers(seller_id), shipping_limit_date TIMESTAMP,\n price NUMERIC(12,2) CHECK(price IS NULL OR price>=0),\n freight_value NUMERIC(12,2) CHECK(freight_value IS NULL OR freight_value>=0),\n PRIMARY KEY(order_id,order_item_id)\n);\nCREATE INDEX idx_hw5_items_product ON olist_order_items(product_id);"
with engine.begin() as conn:
    for stmt in SCHEMA_SQL.split(';'):
        if stmt.strip(): conn.execute(text(stmt))
print('Typed tables + constraints created.')

Typed tables + constraints created.


In [5]:
LOAD_SQL = 'INSERT INTO olist_customers SELECT customer_id,customer_unique_id,customer_zip_code_prefix::INTEGER,customer_city,customer_state::CHAR(2) FROM olist_customers_raw;\nINSERT INTO olist_products SELECT product_id,product_category_name,product_name_lenght::INTEGER,product_description_lenght::INTEGER,product_photos_qty::INTEGER,product_weight_g::INTEGER,product_length_cm::INTEGER,product_height_cm::INTEGER,product_width_cm::INTEGER FROM olist_products_raw;\nINSERT INTO olist_sellers SELECT seller_id,seller_zip_code_prefix::INTEGER,seller_city,seller_state::CHAR(2) FROM olist_sellers_raw;\nINSERT INTO olist_orders SELECT order_id,customer_id,order_status,order_purchase_timestamp::TIMESTAMP,order_approved_at::TIMESTAMP,order_delivered_carrier_date::TIMESTAMP,order_delivered_customer_date::TIMESTAMP,order_estimated_delivery_date::TIMESTAMP FROM olist_orders_raw;\nINSERT INTO olist_order_items SELECT order_id,order_item_id::INTEGER,product_id,seller_id,shipping_limit_date::TIMESTAMP,price::NUMERIC(12,2),freight_value::NUMERIC(12,2) FROM olist_order_items_raw;'
with engine.begin() as conn:
    for stmt in LOAD_SQL.split(';'):
        if stmt.strip(): print('Inserted:',conn.execute(text(stmt)).rowcount)

Inserted: 99441
Inserted: 32951
Inserted: 3095
Inserted: 99441
Inserted: 112650


In [6]:
smoke = query("""
SELECT 'customers' AS table_name,(SELECT COUNT(*) FROM olist_customers_raw) AS raw_n, COUNT(*) AS typed_n FROM olist_customers
UNION ALL SELECT 'orders',(SELECT COUNT(*) FROM olist_orders_raw),COUNT(*) FROM olist_orders
UNION ALL SELECT 'order_items',(SELECT COUNT(*) FROM olist_order_items_raw),COUNT(*) FROM olist_order_items
UNION ALL SELECT 'products',(SELECT COUNT(*) FROM olist_products_raw),COUNT(*) FROM olist_products
UNION ALL SELECT 'sellers',(SELECT COUNT(*) FROM olist_sellers_raw),COUNT(*) FROM olist_sellers
ORDER BY table_name
""")
display(smoke)
assert (smoke.raw_n == smoke.typed_n).all(), 'Втрачено рядки під час типізації'
assert int(smoke.loc[smoke.table_name=='orders','typed_n'].iloc[0]) > 90000, 'Очікується повний Olist, а не stub'
print('SMOKE TEST PASSED — typed/raw counts match.')

,table_name,raw_n,typed_n
0,customers,99441,99441
1,order_items,112650,112650
2,orders,99441,99441
3,products,32951,32951
4,sellers,3095,3095


SMOKE TEST PASSED — typed/raw counts match.


**Data quality та grain.** `customer_id` — ідентифікатор запису клієнта для замовлення, а не гарантовано унікальна людина; `customer_unique_id` позначає покупця між замовленнями. У цьому ДЗ grain фінальної feature table **customer_id**, як у шаблоні курсу; це означає, що для повторних покупок однієї людини потрібна окрема агрегація за `customer_unique_id` у production ML. `order_total` включає ціну товарів і доставку. Порожні замовлення отримують 0 через LEFT JOIN/COALESCE.

In [7]:
quality = query("""
SELECT (SELECT COUNT(*) FROM olist_customers) AS customers,
       (SELECT COUNT(DISTINCT customer_unique_id) FROM olist_customers) AS unique_buyers,
       (SELECT COUNT(*) FROM olist_orders WHERE order_status='canceled') AS canceled_orders,
       (SELECT COUNT(*) FROM olist_products WHERE product_category_name IS NULL) AS products_missing_category,
       (SELECT COUNT(*) FROM olist_orders o WHERE NOT EXISTS (SELECT 1 FROM olist_order_items i WHERE i.order_id=o.order_id)) AS orders_without_items
""")
display(quality)

,customers,unique_buyers,canceled_orders,products_missing_category,orders_without_items
0,99441,96096,625,610,775


## Завдання 2. CTE-chain: 7 named stages + stable reference date

In [8]:
PIPELINE_SQL = "WITH\nreference_date AS (\n -- Stage 0: stable historical reference date\n SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date FROM olist_orders\n),\nstage1_orders_total AS (\n -- Stage 1: one row per order; aggregate order items before downstream JOINs\n SELECT o.customer_id,o.order_id,o.order_status,o.order_purchase_timestamp AS order_ts,\n        o.order_purchase_timestamp::DATE AS order_date,\n        COALESCE(SUM(COALESCE(oi.price,0)+COALESCE(oi.freight_value,0)),0)::NUMERIC(12,2) AS order_total\n FROM olist_orders o LEFT JOIN olist_order_items oi ON oi.order_id=o.order_id\n GROUP BY o.customer_id,o.order_id,o.order_status,o.order_purchase_timestamp\n),\nstage2_rfm AS (\n -- Stage 2: recency, frequency, monetary and first purchase per customer_id\n SELECT s.customer_id,(rd.as_of_date-MAX(s.order_date))::INTEGER AS recency_days,\n        COUNT(*)::INTEGER AS frequency,SUM(s.order_total)::NUMERIC(12,2) AS monetary,\n        MIN(s.order_date) AS first_order_date\n FROM stage1_orders_total s CROSS JOIN reference_date rd\n GROUP BY s.customer_id,rd.as_of_date\n),\nstage3_cohort AS (\n -- Stage 3: cohort quarter of the first purchase\n SELECT r.*,DATE_TRUNC('quarter',r.first_order_date)::DATE AS cohort_quarter FROM stage2_rfm r\n),\nstage4_rolling_aov AS (\n -- Stage 4: true 30-day rolling average per customer, latest observation chosen downstream\n SELECT s.customer_id,s.order_id,s.order_ts,s.order_total,\n        AVG(s.order_total) OVER (\n           PARTITION BY s.customer_id ORDER BY s.order_ts\n           RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW\n        )::NUMERIC(12,2) AS avg_order_value_30d,\n        ROW_NUMBER() OVER (PARTITION BY s.customer_id ORDER BY s.order_ts DESC,s.order_id DESC) AS rn_recent\n FROM stage1_orders_total s\n),\nstage5_cancel_rate AS (\n -- Stage 5: fraction of canceled orders per customer\n SELECT customer_id,AVG(CASE WHEN order_status='canceled' THEN 1.0 ELSE 0.0 END)::NUMERIC(6,4) AS cancellation_rate\n FROM stage1_orders_total GROUP BY customer_id\n),\nstage6_category_revenue AS (\n -- Stage 6: category revenue per customer; unknown categories explicit\n SELECT o.customer_id,COALESCE(p.product_category_name,'(unknown)') AS product_category_name,\n        SUM(COALESCE(oi.price,0)+COALESCE(oi.freight_value,0))::NUMERIC(12,2) AS category_revenue\n FROM olist_orders o JOIN olist_order_items oi ON oi.order_id=o.order_id\n LEFT JOIN olist_products p ON p.product_id=oi.product_id\n GROUP BY o.customer_id,COALESCE(p.product_category_name,'(unknown)')\n),\nstage7_preferred_category AS (\n -- Stage 7: deterministic top-1 category by revenue\n SELECT customer_id,product_category_name,category_revenue\n FROM (SELECT cr.*,ROW_NUMBER() OVER (\n       PARTITION BY cr.customer_id ORDER BY cr.category_revenue DESC,cr.product_category_name\n       ) AS rn FROM stage6_category_revenue cr) ranked\n WHERE rn=1\n)\nSELECT c.customer_id,c.customer_state,\n       COALESCE(coh.recency_days,9999) AS recency_days,\n       COALESCE(coh.frequency,0) AS frequency,\n       COALESCE(coh.monetary,0)::NUMERIC(12,2) AS monetary,\n       coh.first_order_date,coh.cohort_quarter,\n       COALESCE(aov.avg_order_value_30d,0)::NUMERIC(12,2) AS avg_order_value_30d_recent,\n       COALESCE(cancel.cancellation_rate,0)::NUMERIC(6,4) AS cancellation_rate,\n       COALESCE(pc.product_category_name,'(none)') AS preferred_category,\n       COALESCE(pc.category_revenue,0)::NUMERIC(12,2) AS preferred_category_revenue\nFROM olist_customers c\nLEFT JOIN stage3_cohort coh ON coh.customer_id=c.customer_id\nLEFT JOIN stage4_rolling_aov aov ON aov.customer_id=c.customer_id AND aov.rn_recent=1\nLEFT JOIN stage5_cancel_rate cancel ON cancel.customer_id=c.customer_id\nLEFT JOIN stage7_preferred_category pc ON pc.customer_id=c.customer_id"
features_sample=query(PIPELINE_SQL + ' ORDER BY monetary DESC NULLS LAST, customer_id LIMIT 20')
display(features_sample)
assert features_sample.customer_id.is_unique
print('CTE stages: 0–7; features:',list(features_sample.columns))

,customer_id,customer_state,recency_days,frequency,monetary,first_order_date,cohort_quarter,avg_order_value_30d_recent,cancellation_rate,preferred_category,preferred_category_revenue
0,1617b1357756262bfa56ab541c47bc16,RJ,384,1,13664.08,2017-09-29,2017-07-01,13664.08,0.0,telefonia_fixa,13664.08
1,ec5b2ba62e574342386871631fafd3fc,ES,95,1,7274.88,2018-07-15,2018-07-01,7274.88,0.0,telefonia_fixa,7274.88
2,c6e2731c5b391845f6800c97401a43a9,MS,613,1,6929.31,2017-02-12,2017-01-01,6929.31,0.0,utilidades_domesticas,6929.31
3,f48d464a0baaea338cb25f816991ab1f,ES,85,1,6922.21,2018-07-25,2018-07-01,6922.21,0.0,pcs,6922.21
4,3fd6777bbce08a352fddd04e4a7cc8f6,SP,512,1,6726.66,2017-05-24,2017-04-01,6726.66,0.0,artes,6726.66
5,05455dfa7cd02f13d132aa7a6a9729c6,MG,328,1,6081.54,2017-11-24,2017-10-01,6081.54,0.0,agro_industria_e_comercio,6081.54
6,df55c14d1476a9a3467f131269c2477f,RJ,565,1,4950.34,2017-04-01,2017-04-01,4950.34,0.0,eletroportateis,4950.34
7,e0a2412720e9ea4f26c1ac985f6a7358,GO,98,1,4809.44,2018-07-12,2018-07-01,4809.44,1.0,cool_stuff,4809.44
8,24bbf5fd2f2e1b359ee7de94defc4a15,SP,548,1,4764.34,2017-04-18,2017-04-01,4764.34,0.0,eletroportateis,4764.34
9,3d979689f636322c62418b6346b1c6d2,PB,118,1,4681.78,2018-06-22,2018-04-01,4681.78,0.0,pcs,4681.78


CTE stages: 0–7; features: ['customer_id', 'customer_state', 'recency_days', 'frequency', 'monetary', 'first_order_date', 'cohort_quarter', 'avg_order_value_30d_recent', 'cancellation_rate', 'preferred_category', 'preferred_category_revenue']


**Інтерпретація.** `recency_days` визначається від максимуму дати покупки в самому Olist + 1 день, тому відтворюється незалежно від сьогоднішньої дати. `frequency` і `monetary` підраховуються після агрегації позицій до рівня замовлення, що не множить кількість покупок. `avg_order_value_30d_recent` — останнє значення rolling-вікна 30 днів, `cancellation_rate` — частка скасованих замовлень, `preferred_category` — категорія з найбільшим доходом із детермінованим tie-breaker. Для `customer_id` в Olist частота часто дорівнює 1; для реального покупця в production слід агрегувати за `customer_unique_id`.

## Завдання 3. Recursive CTE: referral graph, depth/path/cycle protection

In [9]:
execute("""
CREATE TABLE hw5_referrals (
 referrer_customer_id TEXT NOT NULL REFERENCES olist_customers(customer_id),
 referred_customer_id TEXT NOT NULL REFERENCES olist_customers(customer_id),
 referred_at TIMESTAMPTZ NOT NULL DEFAULT NOW(),
 PRIMARY KEY(referred_customer_id),
 CHECK(referrer_customer_id <> referred_customer_id)
)""")
execute("""
WITH ordered_customers AS (
 SELECT customer_id,ROW_NUMBER() OVER(ORDER BY customer_id) AS rn
 FROM (SELECT customer_id FROM olist_customers ORDER BY customer_id LIMIT 10) x
)
INSERT INTO hw5_referrals(referrer_customer_id,referred_customer_id)
SELECT prev.customer_id,nxt.customer_id
FROM ordered_customers prev JOIN ordered_customers nxt ON nxt.rn=prev.rn+1
ON CONFLICT(referred_customer_id) DO NOTHING
""")
show('SELECT * FROM hw5_referrals ORDER BY referrer_customer_id')

,referrer_customer_id,referred_customer_id,referred_at
0,00012a2ce6f8dcda20d059ce98491703,000161a058600d5901f007fab4c27140,2026-10-27 13:28:18.691479+00:00
1,000161a058600d5901f007fab4c27140,0001fd6190edaaf884bcaf3d49edf079,2026-10-27 13:28:18.691479+00:00
2,0001fd6190edaaf884bcaf3d49edf079,0002414f95344307404f0ace7a26f1d5,2026-10-27 13:28:18.691479+00:00
3,0002414f95344307404f0ace7a26f1d5,000379cdec625522490c315e70c7a9fb,2026-10-27 13:28:18.691479+00:00
4,000379cdec625522490c315e70c7a9fb,0004164d20a9e969af783496f3408652,2026-10-27 13:28:18.691479+00:00
5,0004164d20a9e969af783496f3408652,000419c5494106c306a97b5635748086,2026-10-27 13:28:18.691479+00:00
6,000419c5494106c306a97b5635748086,00046a560d407e99b969756e0b10f282,2026-10-27 13:28:18.691479+00:00
7,00046a560d407e99b969756e0b10f282,00050bf6e01e69d5c0fd612f1bcfb69c,2026-10-27 13:28:18.691479+00:00
8,00050bf6e01e69d5c0fd612f1bcfb69c,000598caf2ef4117407665ac33275130,2026-10-27 13:28:18.691479+00:00


Показано 9 із 9 рядків


,referrer_customer_id,referred_customer_id,referred_at
0,00012a2ce6f8dcda20d059ce98491703,000161a058600d5901f007fab4c27140,2026-10-27 13:28:18.691479+00:00
1,000161a058600d5901f007fab4c27140,0001fd6190edaaf884bcaf3d49edf079,2026-10-27 13:28:18.691479+00:00
2,0001fd6190edaaf884bcaf3d49edf079,0002414f95344307404f0ace7a26f1d5,2026-10-27 13:28:18.691479+00:00
3,0002414f95344307404f0ace7a26f1d5,000379cdec625522490c315e70c7a9fb,2026-10-27 13:28:18.691479+00:00
4,000379cdec625522490c315e70c7a9fb,0004164d20a9e969af783496f3408652,2026-10-27 13:28:18.691479+00:00
5,0004164d20a9e969af783496f3408652,000419c5494106c306a97b5635748086,2026-10-27 13:28:18.691479+00:00
6,000419c5494106c306a97b5635748086,00046a560d407e99b969756e0b10f282,2026-10-27 13:28:18.691479+00:00
7,00046a560d407e99b969756e0b10f282,00050bf6e01e69d5c0fd612f1bcfb69c,2026-10-27 13:28:18.691479+00:00
8,00050bf6e01e69d5c0fd612f1bcfb69c,000598caf2ef4117407665ac33275130,2026-10-27 13:28:18.691479+00:00


In [10]:
RECURSIVE_SQL = "WITH RECURSIVE referral_depth AS (\n -- Anchor: root customers, not referred by anyone\n SELECT c.customer_id,1 AS depth,ARRAY[c.customer_id]::TEXT[] AS path_ids,c.customer_id::TEXT AS path_text\n FROM olist_customers c\n WHERE NOT EXISTS(SELECT 1 FROM hw5_referrals r WHERE r.referred_customer_id=c.customer_id)\n UNION ALL\n -- Recursive: traverse edges; path membership and depth guard stop cycles\n SELECT r.referred_customer_id,rd.depth+1,\n        rd.path_ids || r.referred_customer_id,\n        rd.path_text || ' -> ' || r.referred_customer_id\n FROM hw5_referrals r JOIN referral_depth rd ON rd.customer_id=r.referrer_customer_id\n WHERE NOT r.referred_customer_id=ANY(rd.path_ids) AND rd.depth<50\n)\nSELECT depth,COUNT(*) AS n_customers,MAX(path_text) AS sample_path\nFROM referral_depth GROUP BY depth ORDER BY depth"
referral_summary=show(RECURSIVE_SQL,60)
assert referral_summary.depth.max()==10
assert int(referral_summary.n_customers.sum())==int(query('SELECT COUNT(*) n FROM olist_customers').n.iloc[0])
print('Recursive CTE coverage and depth OK.')

,depth,n_customers,sample_path
0,1,99432,ffffe8b65bbe3087b653a978c870db99
1,2,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
2,3,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
3,4,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
4,5,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
5,6,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
6,7,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
7,8,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
8,9,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
9,10,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...


Показано 10 із 10 рядків
Recursive CTE coverage and depth OK.


**Інтерпретація.** Anchor використовує `NOT EXISTS`, щоб визначити кореневих клієнтів. Recursive part нарощує `depth` і `path_text`. Якщо з’явиться цикл `A → B → A`, перевірка `NOT next_id = ANY(path_ids)` заборонить повторно відвідати вузол, а `depth < 50` обмежить глибину. Однак компонента графа, яка є **повністю циклічною без кореня**, не потрапить до результату root-based traversal: для production потрібен окремий аудит циклів (або `CYCLE` у PostgreSQL). Дані referral — **синтетичні**, не реальна властивість Olist.

## Завдання 4. Materialized View: full pipeline, UNIQUE INDEX, CONCURRENT REFRESH

In [11]:
execute('CREATE MATERIALIZED VIEW mv_olist_customer_features AS ' + PIPELINE_SQL)
execute('CREATE UNIQUE INDEX idx_mv_olist_customer_features_customer_id ON mv_olist_customer_features(customer_id)')
mv_count=int(query('SELECT COUNT(*) AS n FROM mv_olist_customer_features').n.iloc[0])
customer_count=int(query('SELECT COUNT(*) AS n FROM olist_customers').n.iloc[0])
assert mv_count==customer_count, (mv_count,customer_count)
assert int(query('SELECT COUNT(*) n FROM (SELECT customer_id FROM mv_olist_customer_features GROUP BY customer_id HAVING COUNT(*)>1) d').n.iloc[0])==0
print('MV rows:',mv_count,'; UNIQUE index and grain verified')
show('SELECT * FROM mv_olist_customer_features ORDER BY monetary DESC NULLS LAST,customer_id LIMIT 5')

MV rows: 99441 ; UNIQUE index and grain verified


,customer_id,customer_state,recency_days,frequency,monetary,first_order_date,cohort_quarter,avg_order_value_30d_recent,cancellation_rate,preferred_category,preferred_category_revenue
0,1617b1357756262bfa56ab541c47bc16,RJ,384,1,13664.08,2017-09-29,2017-07-01,13664.08,0.0,telefonia_fixa,13664.08
1,ec5b2ba62e574342386871631fafd3fc,ES,95,1,7274.88,2018-07-15,2018-07-01,7274.88,0.0,telefonia_fixa,7274.88
2,c6e2731c5b391845f6800c97401a43a9,MS,613,1,6929.31,2017-02-12,2017-01-01,6929.31,0.0,utilidades_domesticas,6929.31
3,f48d464a0baaea338cb25f816991ab1f,ES,85,1,6922.21,2018-07-25,2018-07-01,6922.21,0.0,pcs,6922.21
4,3fd6777bbce08a352fddd04e4a7cc8f6,SP,512,1,6726.66,2017-05-24,2017-04-01,6726.66,0.0,artes,6726.66


Показано 5 із 5 рядків


,customer_id,customer_state,recency_days,frequency,monetary,first_order_date,cohort_quarter,avg_order_value_30d_recent,cancellation_rate,preferred_category,preferred_category_revenue
0,1617b1357756262bfa56ab541c47bc16,RJ,384,1,13664.08,2017-09-29,2017-07-01,13664.08,0.0,telefonia_fixa,13664.08
1,ec5b2ba62e574342386871631fafd3fc,ES,95,1,7274.88,2018-07-15,2018-07-01,7274.88,0.0,telefonia_fixa,7274.88
2,c6e2731c5b391845f6800c97401a43a9,MS,613,1,6929.31,2017-02-12,2017-01-01,6929.31,0.0,utilidades_domesticas,6929.31
3,f48d464a0baaea338cb25f816991ab1f,ES,85,1,6922.21,2018-07-25,2018-07-01,6922.21,0.0,pcs,6922.21
4,3fd6777bbce08a352fddd04e4a7cc8f6,SP,512,1,6726.66,2017-05-24,2017-04-01,6726.66,0.0,artes,6726.66


In [12]:
# Окремий autocommit connection: CONCURRENTLY не можна запускати всередині транзакційного блоку.
with engine.connect().execution_options(isolation_level='AUTOCOMMIT') as conn:
    conn.execute(text('REFRESH MATERIALIZED VIEW CONCURRENTLY mv_olist_customer_features'))
print('REFRESH MATERIALIZED VIEW CONCURRENTLY — SUCCESS')
assert int(query('SELECT COUNT(*) n FROM mv_olist_customer_features').n.iloc[0])==customer_count

REFRESH MATERIALIZED VIEW CONCURRENTLY — SUCCESS


**Пояснення.** MV зберігає **повний** результат pipeline, включно з rolling AOV, cancellation rate та preferred category. Унікальний індекс на `customer_id` покриває всі рядки та дозволяє `REFRESH ... CONCURRENTLY`. Refresh повторно обчислює ознаки; дані у MV можуть застарівати між оновленнями. Оскільки вихідні дані історичні, розумним вибором є щоденне або on-demand оновлення залежно від бізнес-вимог.

## Завдання 5. EXPLAIN (ANALYZE, BUFFERS): CTE vs MV lookup

In [13]:
def explain(sql):
    with engine.connect() as conn:
        return [row[0] for row in conn.execute(text('EXPLAIN (ANALYZE, BUFFERS) '+sql)).all()]
def plan_ms(lines):
    for line in reversed(lines):
        m=re.search(r'Execution Time:\s*([0-9.]+)\s*ms',line)
        if m: return float(m.group(1))
    return None

# Повне переобчислення всіх features, без LIMIT всередині pipeline.
cte_plan=explain('SELECT * FROM ('+PIPELINE_SQL+') full_features ORDER BY monetary DESC NULLS LAST, customer_id LIMIT 10')
print('=== FULL CTE PIPELINE ===')
print('\n'.join(cte_plan))
cte_ms=plan_ms(cte_plan)
print('Execution Time:',cte_ms,'ms')

=== FULL CTE PIPELINE ===
Limit  (cost=97944.75..97944.78 rows=10 width=154) (actual time=3328.586..3337.166 rows=10 loops=1)
  Buffers: shared hit=12448, temp read=9961 written=10114
  ->  Sort  (cost=97944.75..98155.00 rows=84100 width=154) (actual time=3328.584..3337.161 rows=10 loops=1)
        Sort Key: ((COALESCE(r.monetary, '0'::numeric))::numeric(12,2)) DESC NULLS LAST, c.customer_id
        Sort Method: top-N heapsort  Memory: 26kB
        Buffers: shared hit=12448, temp read=9961 written=10114
        ->  Hash Left Join  (cost=80586.35..96127.38 rows=84100 width=154) (actual time=2400.470..3263.476 rows=99441 loops=1)
              Hash Cond: (c.customer_id = ranked.customer_id)
              Buffers: shared hit=12448, temp read=9961 written=10114
              CTE stage1_orders_total
                ->  Finalize GroupAggregate  (cost=16725.66..27526.12 rows=88592 width=124) (actual time=203.566..524.881 rows=99441 loops=1)
                      Group Key: o_1.order_id
      

In [14]:
# Індексний lookup за customer_id — окремий сценарій, не еквівалентний повному перерахунку.
customer_id=str(query('SELECT customer_id FROM olist_customers ORDER BY customer_id LIMIT 1').customer_id.iloc[0])
lookup_sql='SELECT * FROM mv_olist_customer_features WHERE customer_id = :cid'
with engine.connect() as conn:
    mv_lookup_plan=[r[0] for r in conn.execute(text('EXPLAIN (ANALYZE, BUFFERS) '+lookup_sql),{'cid':customer_id})]
print('=== MV CUSTOMER LOOKUP ===')
print('\n'.join(mv_lookup_plan))
lookup_ms=plan_ms(mv_lookup_plan)
print('Execution Time:',lookup_ms,'ms')

=== MV CUSTOMER LOOKUP ===
Index Scan using idx_mv_olist_customer_features_customer_id on mv_olist_customer_features  (cost=0.42..8.44 rows=1 width=154) (actual time=0.028..0.030 rows=1 loops=1)
  Index Cond: (customer_id = '00012a2ce6f8dcda20d059ce98491703'::text)
  Buffers: shared hit=4
Planning Time: 0.086 ms
Execution Time: 0.049 ms
Execution Time: 0.049 ms


In [15]:
# Додатково — порівняння однакового top-N результату (CTE vs MV).
execute('CREATE INDEX IF NOT EXISTS idx_mv_olist_customer_features_monetary ON mv_olist_customer_features(monetary DESC)')
mv_top_plan=explain('SELECT * FROM mv_olist_customer_features ORDER BY monetary DESC NULLS LAST, customer_id LIMIT 10')
print('=== MV TOP-10 ===')
print('\n'.join(mv_top_plan))
mv_top_ms=plan_ms(mv_top_plan)
print('Execution Time:',mv_top_ms,'ms')
comparison=pd.DataFrame([
 {'scenario':'Full CTE top-10','execution_ms':cte_ms},
 {'scenario':'MV top-10','execution_ms':mv_top_ms},
 {'scenario':'MV customer lookup','execution_ms':lookup_ms}
])
display(comparison)
assert all(v is not None for v in [cte_ms,mv_top_ms,lookup_ms])

=== MV TOP-10 ===
Limit  (cost=4385.01..4386.16 rows=10 width=154) (actual time=31.665..32.681 rows=10 loops=1)
  Buffers: shared hit=1580
  ->  Gather Merge  (cost=4385.01..11111.94 rows=58495 width=154) (actual time=31.663..32.676 rows=10 loops=1)
        Workers Planned: 1
        Workers Launched: 1
        Buffers: shared hit=1580
        ->  Sort  (cost=3385.00..3531.24 rows=58495 width=154) (actual time=28.003..28.004 rows=8 loops=2)
              Sort Key: monetary DESC NULLS LAST, customer_id
              Sort Method: top-N heapsort  Memory: 26kB
              Buffers: shared hit=1580
              Worker 0:  Sort Method: top-N heapsort  Memory: 27kB
              ->  Parallel Seq Scan on mv_olist_customer_features  (cost=0.00..2120.95 rows=58495 width=154) (actual time=0.011..9.836 rows=49720 loops=2)
                    Buffers: shared hit=1536
Planning:
  Buffers: shared hit=15 read=1
Planning Time: 0.209 ms
Execution Time: 32.709 ms
Execution Time: 32.709 ms


,scenario,execution_ms
0,Full CTE top-10,3351.234
1,MV top-10,32.709
2,MV customer lookup,0.049


**Методологія.** Плани містять реальні `Execution Time` та `Buffers` саме цього запуску. Порівняння **CTE top-10 ↔ MV top-10** відповідає одному результату, тоді як MV lookup за `customer_id` демонструє інший, точковий сценарій. Час виконання залежить від кешу, апаратного середовища й планувальника; MV refresh має власну вартість, яка не включена до lookup. Індекс на `monetary` допомагає не всім top-N запитам: `NULLS LAST` і додаткове сортування за `customer_id` можуть змінити план.

## Reflection (250–300 слів, автоматично з фактичного EXPLAIN)

In [16]:
ops=['HashAggregate','Hash Aggregate','GroupAggregate','Group Aggregate','Sort','WindowAgg','Nested Loop','Hash Join','Merge Join','Seq Scan','Index Scan','Bitmap Heap Scan']
cte_found=[op for op in ops if any(op in line for line in cte_plan)]
mv_found=[op for op in ops if any(op in line for line in mv_top_plan)]
lookup_found=[op for op in ops if any(op in line for line in mv_lookup_plan)]
ratio=cte_ms/mv_top_ms if mv_top_ms else float('inf')
reflection=f"""Найскладнішими за логікою є preferred_category та rolling AOV: для першої ознаки потрібно з’єднати замовлення, позиції й товари, згрупувати дохід за категоріями та ранжувати результат; для другої — впорядкувати покупки в межах клієнта і обчислити ковзне вікно 30 днів. За фактичним EXPLAIN для повного CTE-перерахунку в цьому запуску видно такі оператори: {', '.join(cte_found) if cte_found else 'оператори, перелічені у виведеному вище плані'}. Саме операції JOIN, агрегації та сортування є потенційними джерелами витрат, але без профілювання окремих stages не можна чесно назвати один найдорожчим.

Повний CTE-запит для top-10 виконався за {cte_ms:.3f} мс, а top-10 із materialized view — за {mv_top_ms:.3f} мс. Співвідношення часу цього запуску становить {ratio:.2f} раза. Для точкового lookup за customer_id зафіксовано {lookup_ms:.3f} мс; його план містить: {', '.join(lookup_found) if lookup_found else 'оператори, наведені вище'}. Це окремий сценарій, тому не слід безпосередньо прирівнювати його до повного CTE-перерахунку. У плані top-10 із MV присутні: {', '.join(mv_found) if mv_found else 'оператори, наведені вище'}. Результат залежить від кешування та конкретного середовища PostgreSQL.

Materialized view доцільна, коли ознаки багаторазово читаються для inference або звітів, а щоденне оновлення є достатньо свіжим. Для онлайн-операцій, де критична актуальність кожної транзакції, звичайний VIEW може бути правильнішим, оскільки не зберігає застарілу копію. Основний компроміс тут — latency читання проти freshness та recompute-cost; додатково MV використовує дисковий простір.

Рекурсивний CTE обходить referral-граф від коренів. Цикл A → B → A не спричинить нескінченну рекурсію завдяки перевірці відвіданих ID через масив path_ids та обмеженню depth менше 50. Проте повністю циклічний компонент без кореня не буде знайдений таким обходом. Нарешті, Olist містить окремі customer_id для покупок однієї людини: у реальному ML-проєкті слід перейти до customer_unique_id, щоб коректно вимірювати повторні замовлення та уникати хибної інтерпретації frequency."""
word_count=len(reflection.split())
print('Reflection word count:',word_count)
display(Markdown(reflection))
assert 250 <= word_count <= 300, f'Reflection має {word_count} слів, очікується 250–300'

Reflection word count: 261


Найскладнішими за логікою є preferred_category та rolling AOV: для першої ознаки потрібно з’єднати замовлення, позиції й товари, згрупувати дохід за категоріями та ранжувати результат; для другої — впорядкувати покупки в межах клієнта і обчислити ковзне вікно 30 днів. За фактичним EXPLAIN для повного CTE-перерахунку в цьому запуску видно такі оператори: HashAggregate, GroupAggregate, Sort, WindowAgg, Nested Loop, Hash Join, Seq Scan. Саме операції JOIN, агрегації та сортування є потенційними джерелами витрат, але без профілювання окремих stages не можна чесно назвати один найдорожчим.

Повний CTE-запит для top-10 виконався за 3351.234 мс, а top-10 із materialized view — за 32.709 мс. Співвідношення часу цього запуску становить 102.46 раза. Для точкового lookup за customer_id зафіксовано 0.049 мс; його план містить: Index Scan. Це окремий сценарій, тому не слід безпосередньо прирівнювати його до повного CTE-перерахунку. У плані top-10 із MV присутні: Sort, Seq Scan. Результат залежить від кешування та конкретного середовища PostgreSQL.

Materialized view доцільна, коли ознаки багаторазово читаються для inference або звітів, а щоденне оновлення є достатньо свіжим. Для онлайн-операцій, де критична актуальність кожної транзакції, звичайний VIEW може бути правильнішим, оскільки не зберігає застарілу копію. Основний компроміс тут — latency читання проти freshness та recompute-cost; додатково MV використовує дисковий простір.

Рекурсивний CTE обходить referral-граф від коренів. Цикл A → B → A не спричинить нескінченну рекурсію завдяки перевірці відвіданих ID через масив path_ids та обмеженню depth менше 50. Проте повністю циклічний компонент без кореня не буде знайдений таким обходом. Нарешті, Olist містить окремі customer_id для покупок однієї людини: у реальному ML-проєкті слід перейти до customer_unique_id, щоб коректно вимірювати повторні замовлення та уникати хибної інтерпретації frequency.

In [17]:
# Фінальні автоматичні перевірки.
assert (smoke.raw_n==smoke.typed_n).all()
assert mv_count==customer_count
assert len(features_sample)==20
assert referral_summary.depth.max()==10
assert all(x is not None for x in (cte_ms,mv_top_ms,lookup_ms))
assert 250<=word_count<=300
print('FINAL CHECK PASSED')

FINAL CHECK PASSED
